

## Section 5: Non-Maximum Suppression (NMS)

### 1. Why do we need NMS?

Imagine an object detection model detects a car in an image.

Instead of predicting one bounding box around the car, the model might predict five overlapping bounding boxes around the same car.

![YOLO Annotation Guide | CVAT Tutorial | CVAT](https://images.openai.com/static-rsc-4/vNxz2OLz4ECerlmeNzUXg_HWjvGQPw8dEj3ZlY_MZ1TUqc8C7J-kACw-3H3DWh-d8--YcCVA7Dxc7-SBl4StnRSarnBoEXDjwMmqBIYdRebvg9YfahMYgIGFw4r6RyQg5a7vORL0SWgXAzbmrTzj0h4Co12f_z7ZkjHwUYFLAgo?purpose=inline)

[cvat.ai](https://www.cvat.ai/resources/blog/yolo-object-detection?utm_source=chatgpt.com)

Why does this happen?

- The model generates multiple candidate bounding boxes.
- Several candidates may identify the same object.
- Each candidate has a different confidence score.
- We need to select the most suitable box and remove redundant predictions.

**Non-Maximum Suppression (NMS)** solves this problem.

**Definition:** Non-Maximum Suppression is a post-processing algorithm that removes redundant, overlapping bounding boxes while retaining the highest-confidence detections.

### 2. How does NMS work?

Your syllabus gives this pipeline:

```text
Candidate Boxes
      ↓
Confidence Ranking
      ↓
IoU Comparison
      ↓
Suppress Duplicates
      ↓
Final Detections
```

Let's break down each step.

#### Step 1 — Candidate Boxes

The object detection model generates multiple candidate bounding boxes.

For example, it detects the same car using five boxes:

![NMS(Non Max Suppression) | VISIONHONG](https://images.openai.com/static-rsc-4/8TsHBfkExOiLFYEhJk6J7Qdi7Wdih7t_jKTSj6_C_t_1UExmvsbH0N325vxET-Mm6F5zvIJ1ihY2SYUABZDCWNoVy2uaeGCDgvBPXIIvdTdOSz5_9XXLjAvDB4RCPF0JQJnrDb_frjc_BqBiJ8vJfodDgyRHUE8QeJQZfVrgiiI?purpose=inline)

[visionhong.com](https://www.visionhong.com/blog/NMS?utm_source=chatgpt.com)

Each box contains:

- Bounding box coordinates.
- Predicted class.
- Confidence score.

Example:

| Box | Confidence |
|---|---:|
| A | 0.95 |
| B | 0.88 |
| C | 0.82 |
| D | 0.65 |
| E | 0.42 |

These are candidate detections, not necessarily five different cars.

#### Step 2 — Confidence Ranking

NMS sorts all candidate boxes in descending order of confidence.

| Ranking | Box | Confidence |
|---:|---|---:|
| 1 | A | 0.95 |
| 2 | B | 0.88 |
| 3 | C | 0.82 |
| 4 | D | 0.65 |
| 5 | E | 0.42 |

The highest-confidence box is selected first.

In this example, Box A (0.95) becomes our reference box.

#### Step 3 — IoU Comparison

Now compare the selected box with the remaining candidate boxes using Intersection over Union (IoU).

Remember:

$$
IoU=\frac{\text{Intersection Area}}{\text{Union Area}}
$$

Suppose we use an IoU threshold of `0.50`.

| Comparison | IoU | Decision |
|---|---:|---|
| A vs B | 0.82 | Suppress B |
| A vs C | 0.68 | Suppress C |
| A vs D | 0.25 | Keep D |
| A vs E | 0.10 | Keep E |

**Rule:**

- If IoU is greater than the threshold → suppress the overlapping box.
- If IoU is less than or equal to the threshold → keep the box.

Why?

Boxes B and C overlap heavily with A, suggesting they are duplicate predictions of the same object.

Boxes D and E have less overlap, so they remain candidates.

#### Step 4 — Suppress Duplicates

We remove boxes B and C.

Remaining boxes:

| Box | Confidence | Status |
|---|---:|---|
| A | 0.95 | Retained |
| B | 0.88 | Suppressed |
| C | 0.82 | Suppressed |
| D | 0.65 | Retained |
| E | 0.42 | Retained |

Now NMS selects the next highest-confidence remaining box, D.

It compares D against E. If their IoU is greater than 0.50, E is suppressed; otherwise, E remains.

This process repeats until there are no candidate boxes left.

#### Step 5 — Final Detections

![Non-Maximum Suppression with OpenCV and Python - The Python Code](https://images.openai.com/static-rsc-4/4bem5PNIIzlkqN7Qn1imXdp0oa_-IHKjgVRV_cQjUpXsg9FqdsvviX8G9Ldkmay086UpjSnrczDZ9JJO_D9NzQ78G6v6480UAUhOB9IdmBhnsNnxiprLYDpsVgmBqS7NdqSZME6GILq0oLHWDfP1r5DSKJ7NV9Lnb6ncP6kAFVg?purpose=inline)

[thepythoncode.com](https://thepythoncode.com/article/non-maximum-suppression-using-opencv-in-python?utm_source=chatgpt.com)

The final output contains the retained detections rather than all the overlapping predictions.

### 3. NMS Algorithm — Complete Logic

Here is the algorithm in pseudocode:

```text
1. Sort all bounding boxes by confidence score.

2. Select the box with the highest confidence.

3. Keep that box.

4. Calculate IoU between the selected box
   and all remaining boxes.

5. Suppress boxes whose IoU exceeds
   the predefined threshold.

6. Select the next highest-confidence
   remaining box.

7. Repeat until no boxes remain.

8. Return the retained bounding boxes.
```

### 4. Understanding the IoU Threshold in NMS

The IoU threshold controls how aggressively NMS removes overlapping boxes.

| IoU Threshold | Behaviour |
|---:|---|
| 0.30 | More aggressive suppression |
| 0.50 | Moderate suppression |
| 0.70 | Less aggressive suppression |
| 0.90 | Very little suppression |

![An overview of object detection: one-stage methods.](https://images.openai.com/static-rsc-4/LkBLf4CHp1Qy6g2Xh-lj2DoeZFHRb-j4Uihxk8MUuIP9Vu4AK7tThmHQIvj5S942PjPaCLaO8bBL2KChItR7L5I2uWUMLMsbpFHfspi_wyCUCokYpJUOxu83MpRpckS9M2PZu-J6B79ZCEjY3Dt3TP6fGATeeFB6omwj4T_0kS4?purpose=inline)

[jeremyjordan.me](https://www.jeremyjordan.me/object-detection-one-stage/?utm_source=chatgpt.com)

**Important distinction:**

- **Confidence threshold:** Determines whether a prediction is confident enough to consider.
- **IoU threshold:** Determines whether two boxes overlap enough for one to be suppressed.

These are two different parameters!

### 5. Implementing NMS in Python

Let's first understand the basic implementation using NumPy.

We will use boxes in XYXY format:

`[x1, y1, x2, y2]`

#### NMS implementation

```python
import numpy as np


def nms(boxes, scores, iou_threshold=0.5):

    # Sort boxes by confidence (highest first)
    order = np.argsort(scores)[::-1]

    keep = []

    while len(order) > 0:

        # Select highest confidence box
        current = order[0]
        keep.append(current)

        remaining = order[1:]

        if len(remaining) == 0:
            break

        # Calculate IoU with remaining boxes
        ious = np.array([
            calculate_iou(boxes[current], boxes[i])
            for i in remaining
        ])

        # Keep boxes with IoU <= threshold
        order = remaining[ious <= iou_threshold]

    return keep
```

Here, `calculate_iou()` is the function you implemented in the previous section.

Let's test it:

```python
boxes = np.array([
    [100, 100, 300, 300],  # Box A
    [120, 120, 310, 310],  # Box B
    [400, 400, 550, 550]   # Box C
])

scores = np.array([
    0.95,
    0.88,
    0.75
])

selected = nms(boxes, scores, 0.5)

print("Selected boxes:", selected)
```

**Expected output:**

```text
Selected boxes: [0, 2]
```

**Explanation:**

- Box A has the highest confidence, so it is retained.
- Box B overlaps heavily with A, so it is suppressed.
- Box C has low overlap with A, so it is retained.

This is a basic greedy NMS implementation for valid XYXY boxes. It does not include class-aware filtering or a separate confidence cutoff.

### 6. Where is NMS used in industry?

NMS is used in many object detection applications.

#### Autonomous vehicles

![Autonomous vehicle detection](https://images.openai.com/static-rsc-4/QBZoEUaUSa1GmsYpJjOT5ef3Czwuf8byAk4ZjR-YCe0pE3jUHKIcHVGsKEtjt32LY1wQD_in7HY5ouFDGMtTgcDadsB5FFthuF7jrCIPe85qSqFU1l68IdOPnwEok9VKqS7ex1aZBKPJmKzr2GcgZQpp_5whySQmHOzlMSjWbNM?purpose=inline)

Removes duplicate detections of vehicles, pedestrians and other road objects.

#### Surveillance systems

![Pedestrian detection and counting](https://images.openai.com/static-rsc-4/myw4S0-tsqXs4llvfbmRdmkNRPECfOsduNusxQmXwj3LWgW5MooO-HkODR2bBXwPQzqKU9cksQCIzQgCPmyWgCW4WoqRKwz7u1n-8qh7mPljsxRyJC9RDLfrDbAEjc8b1Dh1D4tfI81D5BduycgN4JmRJj82wqHMsTIuxJ9BX7I?purpose=inline)

Helps eliminate redundant person detections.

#### Industrial inspection

![Computer Vision's Impacts in 60 Applications | Ultralytics](https://images.openai.com/static-rsc-4/aLny9CcNHqZvP02HVW5BunqBsKkUTBHzdzvd7Um4I7YE10PgFf1oI0Dd7gK3uH5TI9dHrtrTfzDZ_6xQkl0qF1A76V3jCfLSH5pL-4dr9gKUp9ZvsUAjG1F1AhhN4RdH1fDyKyv7aaKsiHoMf441lkbgOQ3Pl1QdXiO6EKPa978?purpose=inline)

Helps produce cleaner detections of products and defects.

## What You Should Remember

1. NMS stands for Non-Maximum Suppression.
2. It is a post-processing algorithm used in object detection.
3. It ranks candidate boxes by confidence.
4. It uses IoU to identify highly overlapping boxes.
5. It retains selected high-confidence boxes and suppresses redundant ones.
6. A lower IoU threshold means more aggressive suppression.
7. Confidence threshold and IoU threshold serve different purposes.

---